In [1]:
%cd /Users/akarsh/Documents/projects/fastInfer/StoryForge

/Users/akarsh/Documents/projects/fastInfer/StoryForge


In [2]:
import copy
import hashlib
import json
from pathlib import Path

import numpy as np
import torch
from tokenizers import Tokenizer

from storyforge.config import load_config
from storyforge.training import load_initial_model
from storyforge.optimization import build_optimizer
from storyforge.evaluation import evaluate
from storyforge.generation import generate
from storyforge.metric import record_metrics
from storyforge.checkpoint import save_checkpoint
from storyforge.runner import run_training, synchronize
from storyforge.plotting import plot_loss_curves
from storyforge.memory import MPSMemoryMonitor

In [3]:
config = load_config("configs/story_masked_40m.toml")

assert config["model"]["story_boundary_masking"] is True

metrics_path = Path(config["logging"]["metrics_path"])
samples_path = Path(config["generation"]["samples_path"])
best_path = Path(config["checkpoints"]["best_path"])

existing_records = []

if metrics_path.exists():
    with metrics_path.open(encoding="utf-8") as file:
        existing_records = [
            json.loads(line)
            for line in file
            if line.strip()
        ]

    if (
        len(existing_records) != 1
        or existing_records[0]["step"] != 0
        or existing_records[0]["processed_tokens"] != 0
    ):
        raise RuntimeError(
            "This experiment already contains training records. "
            "This notebook starts from initial weights; "
            "it does not resume an interrupted run."
        )

In [4]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

model = load_initial_model(config, device)
optimizer = build_optimizer(model, config)

tokenizer = Tokenizer.from_file(
    config["tokenizer"]["path"]
)

train_tokens = np.load(
    config["encoding"]["train_path"],
    mmap_mode="r",
)

val_tokens = np.load(
    config["encoding"]["validation_path"],
    mmap_mode="r",
)

rng = np.random.default_rng(config["seed"])

tokenizer_hash = hashlib.sha256(
    Path(config["tokenizer"]["path"]).read_bytes()
).hexdigest()

print("Device:", device)
print("Story-boundary masking:", model.story_boundary_masking)
print("Starting weights:", config["checkpoints"]["initial_path"])

Device: mps
Story-boundary masking: True
Starting weights: runs/baseline/initial.pt


In [5]:
if samples_path.exists():
    with samples_path.open(encoding="utf-8") as file:
        sample_history = json.load(file)

    if set(sample_history) != {"initial"}:
        raise RuntimeError(
            "This samples file already contains later checkpoints."
        )

    initial = sample_history["initial"]

    if initial["processed_tokens"] != 0:
        raise ValueError("Expected zero-token initial samples.")

    if initial["tokenizer_sha256"] != tokenizer_hash:
        raise ValueError("Initial samples used a different tokenizer.")

    for key in (
        "prompts",
        "max_new_tokens",
        "temperature",
        "top_k",
        "seed",
    ):
        if (
            initial["generation_config"][key]
            != config["generation"][key]
        ):
            raise ValueError(
                f"Initial generation setting differs: {key}"
            )

    if len(initial["samples"]) != len(config["generation"]["prompts"]):
        raise ValueError("Unexpected number of initial samples.")

    for index, sample in enumerate(initial["samples"]):
        expected_prompt = config["generation"]["prompts"][index]
        expected_seed = config["generation"]["seed"] + index

        if (
            sample["prompt"] != expected_prompt
            or sample["seed"] != expected_seed
        ):
            raise ValueError("Initial prompt or sampling seed differs.")

    print("Reusing saved initial samples.")

else:
    initial_samples = []

    for index, prompt in enumerate(config["generation"]["prompts"]):
        seed = config["generation"]["seed"] + index

        text = generate(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            config=config,
            device=device,
            seed=seed,
        )

        initial_samples.append({
            "prompt": prompt,
            "seed": seed,
            "text": text,
        })

    sample_history = {
        "initial": {
            "processed_tokens": 0,
            "tokenizer_sha256": tokenizer_hash,
            "generation_config": copy.deepcopy(
                config["generation"]
            ),
            "samples": initial_samples,
        },
    }

    samples_path.parent.mkdir(parents=True, exist_ok=True)

    with samples_path.open("w", encoding="utf-8") as file:
        json.dump(
            sample_history,
            file,
            indent=2,
            ensure_ascii=False,
        )

    print("Saved initial samples:", samples_path)


Saved initial samples: runs/story_masked_40m/samples.json


In [6]:
if existing_records:
    initial_validation_loss = existing_records[0]["val_loss"]

    if initial_validation_loss is None:
        raise ValueError("Initial validation loss is missing.")

    print("Reusing saved initial validation measurement.")

else:
    initial_validation_loss = evaluate(
        model=model,
        val_tokens=val_tokens,
        config=config,
        device=device,
    )

    record_metrics(
        metrics_path,
        {
            "step": 0,
            "processed_tokens": 0,
            "train_loss": None,
            "val_loss": initial_validation_loss,
            "learning_rate": None,
            "elapsed_seconds": 0.0,
        },
    )

if not best_path.exists():
    save_checkpoint(
        path=best_path,
        model=model,
        optimizer=optimizer,
        config=config,
        step=0,
        processed_tokens=0,
        val_loss=initial_validation_loss,
    )

print("Initial validation loss:", initial_validation_loss)


Initial validation loss: 9.10287743806839


In [7]:
monitor = None
memory_result = None
run_completed = False

if device.type == "mps":
    monitor = MPSMemoryMonitor(
        config["profiling"]["sample_interval_seconds"]
    )

    synchronize(device)
    monitor.start()

    print("MPS memory monitoring enabled.")

elif device.type == "cuda":
    print("CUDA peak-memory measurement enabled by the runner.")

else:
    print("Running on CPU; no GPU memory measurement is available.")

try:
    summary = run_training(
        model=model,
        optimizer=optimizer,
        train_tokens=train_tokens,
        val_tokens=val_tokens,
        config=config,
        rng=rng,
        device=device,
        initial_validation_loss=initial_validation_loss,
    )

    run_completed = True

finally:
    if monitor is not None:
        try:
            synchronize(device)
        finally:
            memory_result = monitor.stop()

        memory_result.update({
            "measurement_type": "sampled MPS memory maxima",
            "scope": (
                "main training loop, including periodic validation "
                "and generation; excludes initial setup"
            ),
            "run_completed": run_completed,
            "processed_tokens": (
                summary["processed_tokens"]
                if run_completed
                else None
            ),
            "parameter_dtype": str(
                next(model.parameters()).dtype
            ),
        })

        memory_path = Path(
            config["profiling"]["output_path"]
        )

        memory_path.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        with memory_path.open("w", encoding="utf-8") as file:
            json.dump(memory_result, file, indent=2)

        print("Saved memory measurements:", memory_path)



MPS memory monitoring enabled.
Update 1/2442 | tokens 16,384 | train 9.0761 | validation None
Update 10/2442 | tokens 163,840 | train 8.3871 | validation None
Update 20/2442 | tokens 327,680 | train 7.9168 | validation None
Update 30/2442 | tokens 491,520 | train 7.4259 | validation None
Update 40/2442 | tokens 655,360 | train 6.8592 | validation None
Update 50/2442 | tokens 819,200 | train 6.3159 | validation None
Update 60/2442 | tokens 983,040 | train 5.7309 | validation None
Update 70/2442 | tokens 1,146,880 | train 5.3234 | validation None
Update 80/2442 | tokens 1,310,720 | train 4.9971 | validation None
Update 90/2442 | tokens 1,474,560 | train 4.6841 | validation None
Update 100/2442 | tokens 1,638,400 | train 4.4982 | validation 4.167628586292267
Update 110/2442 | tokens 1,802,240 | train 4.3462 | validation None
Update 120/2442 | tokens 1,966,080 | train 4.1946 | validation None
Update 130/2442 | tokens 2,129,920 | train 4.0987 | validation None
Update 140/2442 | tokens 2,293

In [8]:
if memory_result is not None:
    summary["memory_profile"] = memory_result

    summary_path = Path(
        config["logging"]["summary_path"]
    )

    with summary_path.open("w", encoding="utf-8") as file:
        json.dump(summary, file, indent=2)


In [9]:
plot_path = plot_loss_curves(config)

print("\nTraining completed.")
print("Processed targets:", summary["processed_tokens"])
print("Best validation loss:", summary["best_val_loss"])
print("Best validation perplexity:", summary["best_val_perplexity"])
print("Training tokens/second:", summary["training_tokens_per_second"])
print("Loss plot:", plot_path)

if memory_result is not None:
    print("\nMPS memory measurements:")
    print(json.dumps(memory_result, indent=2))

elif device.type == "cuda":
    print(
        "Peak CUDA tensor allocation:",
        summary["peak_cuda_allocated_gib"],
        "GiB",
    )
    print(
        "Peak CUDA reserved memory:",
        summary["peak_cuda_reserved_gib"],
        "GiB",
    )


Training completed.
Processed targets: 40009728
Best validation loss: 1.9416252672672272
Best validation perplexity: 6.970069996540188
Training tokens/second: 28987.676656759402
Loss plot: runs/story_masked_40m/loss_curves.png

MPS memory measurements:
{
  "max_observed_tensor_gib": 1.0311152935028076,
  "max_observed_driver_gib": 2.1131134033203125,
  "sample_count": 114074,
  "requested_sample_interval_seconds": 0.01,
  "measurement_type": "sampled MPS memory maxima",
  "scope": "main training loop, including periodic validation and generation; excludes initial setup",
  "run_completed": true,
  "processed_tokens": 40009728,
  "parameter_dtype": "torch.float32"
}
